# sigstore_models smoke test

Pydantic v2 models of the Sigstore protobuf specs (`bundle`, `common`, `rekor`, `trustroot`,
`intoto`) by astral-sh. Pure python. The models enforce strict protobuf-JSON semantics:
`bytes` fields take/emit base64, `uint64` fields take/emit decimal strings, and wire keys are
camelCase aliases. Everything below is offline — no Sigstore infrastructure is contacted.

In [ ]:
import base64
from sigstore_models.common.v1 import LogId

log_id = LogId(key_id=base64.b64encode(b"\x01\x02\x03\x04"))
as_json = log_id.to_json()
assert '"keyId"' in as_json  # snake_case field -> camelCase wire key
as_dict = log_id.to_dict()
assert as_dict["keyId"] == "AQIDBA=="  # bytes -> base64 string

back = LogId.from_json(as_json)
assert back == log_id and back.key_id == b"\x01\x02\x03\x04"
print("wire form:", as_json)

In [ ]:
import base64
from sigstore_models.rekor.v1 import TransparencyLogEntry

b64 = lambda raw: base64.b64encode(raw).decode()
entry = TransparencyLogEntry.from_dict({
    "logIndex": "42",
    "logId": {"keyId": b64(b"log-key")},
    "kindVersion": {"kind": "hashedrekord", "version": "0.0.1"},
    "integratedTime": "1700000000",
    "inclusionProof": {
        "logIndex": "42",
        "rootHash": b64(b"root-hash"),
        "treeSize": "128",
        "hashes": [b64(b"n0"), b64(b"n1")],
        "checkpoint": {"envelope": "rekor.sigstore.dev - 1193050959916656506\n128\n..."},
    },
    "canonicalizedBody": b64(b"{}"),
})
assert entry.integrated_time == 1700000000 and entry.log_index == 42  # parsed to int
assert entry.inclusion_proof.hashes == [b"n0", b"n1"]  # decoded to raw bytes
print("parsed entry | log index:", entry.log_index, "| tree size:", entry.inclusion_proof.tree_size)

In [ ]:
import base64, hashlib
from sigstore_models.bundle.v1 import Bundle, VerificationMaterial
from sigstore_models.common.v1 import HashAlgorithm, HashOutput, MessageSignature, X509Certificate

digest = hashlib.sha256(b"bundle payload").digest()
bundle = Bundle(
    media_type="application/vnd.dev.sigstore.bundle.v0.3+json",
    verification_material=VerificationMaterial(
        certificate=X509Certificate(raw_bytes=base64.b64encode(b"fake-der-certificate")),
        tlog_entries=[],
    ),
    message_signature=MessageSignature(
        message_digest=HashOutput(algorithm=HashAlgorithm.SHA2_256, digest=base64.b64encode(digest)),
        signature=base64.b64encode(b"not-a-real-signature"),
    ),
)
rt = Bundle.from_json(bundle.to_json())
assert rt == bundle
assert rt.verification_material.certificate.raw_bytes == b"fake-der-certificate"
print("round-trip OK:", bundle.media_type)

In [ ]:
import base64
from pydantic import ValidationError
from sigstore_models.common.v1 import HashAlgorithm, HashOutput, LogId
from sigstore_models.rekor.v1 import InclusionProof

try:  # ProtoBytes: 5 base64 chars is an invalid encoding length
    HashOutput(algorithm=HashAlgorithm.SHA2_256, digest=b"abcde")
    raise AssertionError("malformed base64 must be rejected")
except ValidationError as e:
    print("rejected malformed base64:", e.error_count(), "error(s)")

try:  # ProtoU64: protobuf JSON requires uint64 as *string*, not int
    InclusionProof(log_index=7, root_hash=base64.b64encode(b"r"), tree_size="8", hashes=[])
    raise AssertionError("int uint64 must be rejected")
except ValidationError as e:
    print("rejected non-string uint64:", e.error_count(), "error(s)")

try:  # extra="forbid": unknown wire keys are rejected
    LogId.from_dict({"keyId": "AQIDBA==", "unexpected": 1})
    raise AssertionError("unknown key must be rejected")
except ValidationError as e:
    print("rejected unknown key:", e.error_count(), "error(s)")